In [15]:
import mlflow
import pandas as pd


mlflow.set_tracking_uri("sqlite:///../mlflow.db")

In [2]:
from mlflow.tracking import MlflowClient

In [3]:
def select_best_model(experiment_name, metric="f1", model_type='tree', data_version="v1"):
    client = MlflowClient()
    experiment = client.get_experiment_by_name(experiment_name)
    if experiment is None:
        raise ValueError(f"Experiment '{experiment_name}' not found.")

    # Build filter
    filter_str = f"run_name = 'Tuning' AND tags.data_version = '{data_version}'"
    if model_type:
        filter_str += f" AND tags.model_type = '{model_type}'"

    # Search runs
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=filter_str,
        order_by=[f"metrics.{metric} DESC"],
    )

    if not runs:
        raise ValueError(f"No runs found for model_type='{model_type}' and data_version='{data_version}'.")

    # Take best run
    best_run = runs[0]
    run_id = best_run.info.run_id
    
    return best_run

In [5]:
br = select_best_model("network_security")
br

2025/11/05 20:37:47 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/11/05 20:37:47 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


<Run: data=<RunData: metrics={'accuracy': 0.9845417502102063,
 'f1': 0.9893983565766762,
 'precision': 0.9954932999136993,
 'recall': 0.9845417502102063}, params={'balance_factor': '0.2',
 'criterion': 'log_loss',
 'max_depth': '10',
 'min_samples_split': '6',
 'train_shape': '(150434, 78)',
 'val_shape': '(15461, 78)',
 'val_size': '0.2'}, tags={'data_version': 'v1',
 'mlflow.runName': 'Tuning',
 'mlflow.source.git.commit': '4bbd988ca3bc119ed294572010c85e95e5d6f5ad',
 'mlflow.source.name': '/home/marcos/Escritorio/AI-prod/AI-Driven-Network-Security/scripts/tuning.py',
 'mlflow.source.type': 'LOCAL',
 'mlflow.user': 'marcos',
 'model_type': 'tree'}>, info=<RunInfo: artifact_uri='/home/marcos/Escritorio/AI-prod/AI-Driven-Network-Security/mlruns/1/b5b1c2c5d0f540d7b2a3583843645938/artifacts', end_time=1762384892888, experiment_id='1', lifecycle_stage='active', run_id='b5b1c2c5d0f540d7b2a3583843645938', run_name='Tuning', start_time=1762384889406, status='FINISHED', user_id='marcos'>, inpu

In [7]:
import sys
print(sys.executable)


/usr/bin/python3


In [8]:
!/usr/bin/python3 -m pip install -U scikit-learn==1.7.2


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 10.9 MB/s  0:00:00 11.8 MB/s eta 0:00:01
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.1.3
    Uninstalling scikit-learn-1.1.3:
      Successfully uninstalled scikit-learn-1.1.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-models-official 2.15.0 requires tensorflow~=2.15.0, but you have tensorflow 2.20.0 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /usr/bin/python3 -m pip install --upgrade pip


In [1]:
import sklearn
print(sklearn.__version__)


1.7.2


In [6]:
# Run info (from your printed run)
run_id = "b5b1c2c5d0f540d7b2a3583843645938"

# Load the model
model = mlflow.sklearn.load_model(f"runs:/{run_id}/model")

In [34]:
"""
import pandas as pd
import matplotlib.pyplot as plt

# Suppose model is a trained DecisionTreeClassifier
importances = model.feature_importances_


#features = X_train.columns

features = [
    'Flow Duration',
    'Total Fwd Packets',
    'Total Backward Packets',
    'Flow Bytes/s',
    'Flow Packets/s',
]



# Create DataFrame
fi = pd.DataFrame({'feature': features, 'importance': importances})
fi = fi.sort_values('importance', ascending=False)

# Plot
plt.figure(figsize=(8, 5))
plt.barh(fi['feature'], fi['importance'])
plt.gca().invert_yaxis()
plt.title("Feature Importance")
plt.show()
"""

'\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\n# Suppose model is a trained DecisionTreeClassifier\nimportances = model.feature_importances_\n\n\n#features = X_train.columns\n\nfeatures = [\n    \'Flow Duration\',\n    \'Total Fwd Packets\',\n    \'Total Backward Packets\',\n    \'Flow Bytes/s\',\n    \'Flow Packets/s\',\n]\n\n\n\n# Create DataFrame\nfi = pd.DataFrame({\'feature\': features, \'importance\': importances})\nfi = fi.sort_values(\'importance\', ascending=False)\n\n# Plot\nplt.figure(figsize=(8, 5))\nplt.barh(fi[\'feature\'], fi[\'importance\'])\nplt.gca().invert_yaxis()\nplt.title("Feature Importance")\nplt.show()\n'

## Neural network

In [16]:
bnn = select_best_model("network_security", model_type="nn")
bnn

<Run: data=<RunData: metrics={'accuracy': 0.9549188280188863,
 'f1': 0.9637456999309392,
 'precision': 0.9762690276820102,
 'recall': 0.9549188280188863}, params={'balance_factor': '0.2',
 'batch_size': '64',
 'early_stop': '-1',
 'final_lr': '0.0018288939647533012',
 'hidden1': '128',
 'hidden2': '64',
 'lr': '0.0018288939647533012',
 'optimizer': 'Adam',
 'scaler_type': 'standard',
 'train_shape': '(150434, 78)',
 'val_shape': '(15461, 78)',
 'val_size': '0.2'}, tags={'data_version': 'v1',
 'mlflow.runName': 'Tuning',
 'mlflow.source.git.commit': '4bbd988ca3bc119ed294572010c85e95e5d6f5ad',
 'mlflow.source.name': '/home/marcos/Escritorio/AI-prod/AI-Driven-Network-Security/scripts/tuning.py',
 'mlflow.source.type': 'LOCAL',
 'mlflow.user': 'marcos',
 'model_type': 'nn'}>, info=<RunInfo: artifact_uri='/home/marcos/Escritorio/AI-prod/AI-Driven-Network-Security/mlruns/1/5ab1f4e3243d4a3895d99a0784758f88/artifacts', end_time=1762389113104, experiment_id='1', lifecycle_stage='active', run_id

In [17]:
bnn.info

<RunInfo: artifact_uri='/home/marcos/Escritorio/AI-prod/AI-Driven-Network-Security/mlruns/1/5ab1f4e3243d4a3895d99a0784758f88/artifacts', end_time=1762389113104, experiment_id='1', lifecycle_stage='active', run_id='5ab1f4e3243d4a3895d99a0784758f88', run_name='Tuning', start_time=1762389107824, status='FINISHED', user_id='marcos'>

In [18]:
artifact_paths = {
    "preprocessing": "preprocessing",       # folder under artifacts
    "plots": "plots",                        # folder under artifacts
    "model": "model"                         # model artifact path
}

In [19]:
run_id = bnn.info.run_id

In [20]:
run_id

'5ab1f4e3243d4a3895d99a0784758f88'

In [21]:
import sys, os
sys.path.append(os.path.abspath("src"))

sys.path.append(os.path.abspath("."))


In [26]:
import mlflow.pytorch


artifact_paths = {
    "preprocessing": "preprocessing",       # folder under artifacts
    "plots": "plots",                        # folder under artifacts
    "model": "model"                         # model artifact path
}

# --------------------------
model_uri = f"runs:/{run_id}/{artifact_paths['model']}"
model = mlflow.pytorch.load_model(model_uri)
print("TensorFlow model loaded successfully!")






2025/11/05 21:44:10 WARNING mlflow.pytorch: Stored model version '2.8.0+cpu' does not match installed PyTorch version '2.3.1+cpu'


TensorFlow model loaded successfully!


In [24]:
import os, sys
sys.path.append(os.path.abspath(".."))  # if notebook is inside /notebooks


In [25]:
import src
print("Imports OK!")


Imports OK!


In [27]:
print(model)

NNModel(
  (model): Sequential(
    (0): Linear(in_features=78, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)
